# 00_bootstrap_fred

Fetches 5 years of macroeconomic indicator data (2020-01-01 to today) from
the FRED API and writes to `precursor.bronze.fred_raw`.

**Run this notebook ONCE manually.** After bootstrap, `01_ingestion.py`
handles daily updates.

In [0]:
%pip install fredapi --no-deps

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
%pip install "numpy<2.0" --no-deps

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()

In [0]:
import fredapi
import pandas as pd
import logging
from datetime import datetime
from typing import Optional

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    StringType, DateType, DoubleType, TimestampType, BooleanType,
)

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("precursor.fred_bootstrap")

START_DATE = "2020-01-01"
END_DATE   = datetime.today().strftime("%Y-%m-%d")

FRED_SERIES = {
    "DFF":      "Fed Funds Rate (daily)",
    "UNRATE":   "Unemployment Rate (monthly)",
    "CPIAUCSL": "CPI Inflation (monthly)",
    "T10Y2Y":   "10Y-2Y Yield Spread (daily)",
    "VIXCLS":   "VIX Volatility Index (daily)",
    "M2SL":     "M2 Money Supply (monthly)",
}

_DAILY_SERIES  = {"DFF", "T10Y2Y", "VIXCLS"}
_MONTHLY_SERIES = {"UNRATE", "CPIAUCSL", "M2SL"}

_FRED_RAW_SCHEMA = StructType([
    StructField("series_id",        StringType(),    nullable=False),
    StructField("description",      StringType(),    nullable=False),
    StructField("date",             DateType(),      nullable=False),
    StructField("value",            DoubleType(),    nullable=True),
    StructField("frequency",        StringType(),    nullable=False),
    StructField("is_forward_filled", BooleanType(),  nullable=False),
    StructField("ingested_at",      TimestampType(), nullable=False),
])

logger.info("START_DATE=%s  END_DATE=%s", START_DATE, END_DATE)

INFO:precursor.fred_bootstrap:START_DATE=2020-01-01  END_DATE=2026-05-01


## Cell 3 — Load credentials

In [0]:
def get_fred_client() -> fredapi.Fred:
    """Build a FRED client from Databricks secrets.

    The API key is stored in scope="precursor" under key="fred-key".

    Returns:
        Configured fredapi.Fred client.
    """
    api_key = dbutils.secrets.get(scope="precursor", key="fred-key")
    client  = fredapi.Fred(api_key=api_key)
    logger.info("FRED client initialised successfully.")
    return client

INFO:py4j.clientserver:Received command c on object id p0


## Cell 4 — Fetch one FRED series

In [0]:
def fetch_fred_series(
    client: fredapi.Fred,
    series_id: str,
    description: str,
    start_date: str,
    end_date: str,
) -> Optional[pd.DataFrame]:
    """Fetch observations for a single FRED series.

    FRED returns a pandas Series indexed by date. We keep NaN rows as-is
    rather than dropping or filling them here — forward-filling monthly
    series to daily frequency happens in forward_fill_to_daily(), and any
    remaining NaNs in daily series are left for the silver layer to handle.
    Dropping NaNs at this stage would silently lose the date index entries
    that downstream logic needs to detect gaps.

    Args:
        client:      Authenticated fredapi.Fred client.
        series_id:   FRED series identifier (e.g. "DFF").
        description: Human-readable label stored in the table.
        start_date:  Observation window start (YYYY-MM-DD).
        end_date:    Observation window end (YYYY-MM-DD).

    Returns:
        DataFrame with columns [series_id, description, date, value,
        frequency, is_forward_filled, ingested_at], or None on error.
    """
    import time
    for attempt in range(2):
        try:
            raw = client.get_series(
                series_id,
                observation_start=start_date,
                observation_end=end_date,
            )
            break
        except Exception as exc:
            if attempt == 0:
                logger.warning("%s: fetch failed (%s) — retrying in 10s...", series_id, exc)
                time.sleep(10)
            else:
                logger.error("%s: fetch failed after retry — %s", series_id, exc)
                return None

    if raw is None or raw.empty:
        logger.warning("%s: no observations returned.", series_id)
        return None

    frequency = "daily" if series_id in _DAILY_SERIES else "monthly"

    df = pd.DataFrame({
        "series_id":         series_id,
        "description":       description,
        "date":              pd.to_datetime(raw.index),
        "value":             raw.values.astype(float),
        "frequency":         frequency,
        "is_forward_filled": False,
        "ingested_at":       datetime.utcnow(),
    })

    logger.info(
        "%s: %d observations fetched from %s to %s",
        series_id, len(df),
        df["date"].min().date(), df["date"].max().date(),
    )
    return df

## Cell 5 — Forward fill monthly to daily

In [0]:
def forward_fill_to_daily(df: pd.DataFrame) -> pd.DataFrame:
    """Forward-fill a monthly FRED series to every calendar day.

    Monthly macro indicators (unemployment, CPI, M2) are released once per
    month but our feature store needs a value for every calendar day. We do
    this forward fill at the bronze ingestion layer — not in silver — because
    it is a source-level transformation: we are expanding the native FRED
    monthly cadence to daily without joining any other dataset. Silver joins
    should operate on already-daily data so they stay simple.

    The calendar comes from precursor.bronze.universe which already has a
    row per (ticker, date) pair; we take DISTINCT dates from it to get a
    clean daily spine.

    merge_asof fills each calendar day with the most recent monthly
    observation on or before that day, which is the correct economic
    interpretation (the latest known value as of that date).

    Args:
        df: Monthly-frequency DataFrame from fetch_fred_series.

    Returns:
        Daily-frequency DataFrame with is_forward_filled=True for rows that
        did not correspond to an original monthly observation date.
    """
    calendar = (
        spark.sql("""
            SELECT DISTINCT date
            FROM precursor.bronze.universe
            ORDER BY date
        """)
        .toPandas()
    )
    calendar["date"] = pd.to_datetime(calendar["date"])

    df = df.copy()
    df["date"] = pd.to_datetime(df["date"])
    df_sorted  = df.sort_values("date").reset_index(drop=True)
    cal_sorted = calendar.sort_values("date").reset_index(drop=True)

    # merge_asof assigns each calendar day the latest monthly observation
    # on or before that day — forward fill semantics
    merged = pd.merge_asof(cal_sorted, df_sorted, on="date", direction="backward")

    # Mark rows that were not original observation dates
    original_dates           = set(df_sorted["date"])
    merged["is_forward_filled"] = ~merged["date"].isin(original_dates)

    # Drop calendar days before the first observation (no value to forward fill from)
    merged = merged.dropna(subset=["series_id"])

    merged["ingested_at"] = datetime.utcnow()

    logger.info(
        "%s: forward-filled from %d monthly rows to %d daily rows",
        df["series_id"].iloc[0], len(df), len(merged),
    )
    return merged

## Cell 6 — Fetch all series and write to Delta

In [0]:
def fetch_all_series(client: fredapi.Fred) -> pd.DataFrame:
    """Fetch all FRED series and expand monthly ones to daily frequency.

    Args:
        client: Authenticated fredapi.Fred client.

    Returns:
        Combined DataFrame containing all series at daily frequency.
    """
    all_frames: list[pd.DataFrame] = []

    for series_id, description in FRED_SERIES.items():
        logger.info("Fetching %s — %s", series_id, description)

        df = fetch_fred_series(client, series_id, description, START_DATE, END_DATE)
        if df is None:
            logger.warning("%s: skipping — no data returned.", series_id)
            continue

        if series_id in _MONTHLY_SERIES:
            df = forward_fill_to_daily(df)

        all_frames.append(df)

    if not all_frames:
        raise RuntimeError("No FRED series returned data — check API key and series IDs.")

    combined = pd.concat(all_frames, ignore_index=True)
    logger.info("All series fetched: %d total rows across %d series.", len(combined), len(all_frames))
    return combined


def write_fred_to_delta(df: pd.DataFrame) -> None:
    """Convert combined DataFrame to Spark and write to precursor.bronze.fred_raw.

    Args:
        df: Combined DataFrame from fetch_all_series.
    """
    df = df.copy()
    df["date"]        = pd.to_datetime(df["date"])
    df["ingested_at"] = pd.to_datetime(df["ingested_at"])

    # Ensure value column is float (FRED NaNs come through as float already,
    # but explicit cast guards against object dtype on edge cases)
    df["value"] = df["value"].astype(float)

    keep = ["series_id", "description", "date", "value", "frequency", "is_forward_filled", "ingested_at"]
    df   = df[keep]

    spark_df = spark.createDataFrame(df, schema=_FRED_RAW_SCHEMA)

    (
        spark_df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable("precursor.bronze.fred_raw")
    )

    written = spark.table("precursor.bronze.fred_raw").count()
    logger.info("precursor.bronze.fred_raw written: %d rows.", written)

## Cell 7 — Validation

In [0]:
def _check(name: str, passed: bool, detail: str = "") -> tuple[str, bool, str]:
    status = "PASS" if passed else "FAIL"
    msg    = f"[{status}] {name}" + (f" — {detail}" if detail else "")
    logger.info(msg)
    return (name, passed, detail)


def run_validation() -> list[tuple[str, bool, str]]:
    """Run post-write validation checks on precursor.bronze.fred_raw.

    Returns:
        List of (check_name, passed, detail) tuples.
    """
    checks: list[tuple[str, bool, str]] = []

    # 1. Exactly 6 distinct series_ids
    distinct_ids = spark.sql(
        "SELECT COUNT(DISTINCT series_id) AS n FROM precursor.bronze.fred_raw"
    ).collect()[0]["n"]
    checks.append(_check("Exactly 6 distinct series_ids", distinct_ids == 6, f"{distinct_ids} found"))

    # 2. All 6 series have data
    series_counts = spark.sql(
        "SELECT series_id, COUNT(*) AS n FROM precursor.bronze.fred_raw GROUP BY series_id"
    ).toPandas()
    all_have_data = all(series_counts["n"] > 0)
    checks.append(_check(
        "All 6 series have data",
        all_have_data,
        ", ".join(f"{r['series_id']}={r['n']}" for _, r in series_counts.iterrows()),
    ))

    # 3. Latest date is yesterday or today for daily series
    yesterday = (datetime.today()).date()
    for sid in _DAILY_SERIES:
        row = spark.sql(f"""
            SELECT MAX(date) AS max_date
            FROM precursor.bronze.fred_raw
            WHERE series_id = '{sid}' AND is_forward_filled = FALSE
        """).collect()[0]
        latest = row["max_date"]
        # Allow up to 5 days lag (weekends, holidays, FRED publication delay)
        from datetime import timedelta
        lag_ok = latest >= (datetime.today().date() - timedelta(days=5))
        checks.append(_check(f"{sid}: latest date within 5 days", lag_ok, f"latest = {latest}"))

    # 4. No gaps longer than 5 days in daily series
    for sid in _DAILY_SERIES:
        max_gap = spark.sql(f"""
            SELECT MAX(gap_days) AS max_gap
            FROM (
                SELECT DATEDIFF(
                    date,
                    LAG(date) OVER (ORDER BY date)
                ) AS gap_days
                FROM precursor.bronze.fred_raw
                WHERE series_id = '{sid}'
            ) t
            WHERE gap_days IS NOT NULL
        """).collect()[0]["max_gap"]
        checks.append(_check(
            f"{sid}: no gaps > 5 days",
            max_gap is None or max_gap <= 5,
            f"max gap = {max_gap} days",
        ))

    # 5. Monthly series have values for every calendar day after forward fill
    for sid in _MONTHLY_SERIES:
        null_count = spark.sql(f"""
            SELECT COUNT(*) AS n
            FROM precursor.bronze.fred_raw
            WHERE series_id = '{sid}' AND value IS NULL
        """).collect()[0]["n"]
        checks.append(_check(
            f"{sid}: no nulls after forward fill",
            null_count == 0,
            f"{null_count} nulls found",
        ))

    # 6. No null values in daily series
    for sid in _DAILY_SERIES:
        null_count = spark.sql(f"""
            SELECT COUNT(*) AS n
            FROM precursor.bronze.fred_raw
            WHERE series_id = '{sid}' AND value IS NULL
        """).collect()[0]["n"]
        checks.append(_check(
            f"{sid}: no null values",
            null_count == 0,
            f"{null_count} nulls found",
        ))

    # 7. Value ranges are reasonable
    range_checks = {
        "DFF":    (0, 25),
        "UNRATE": (0, 25),
        "VIXCLS": (0, 100),
        "T10Y2Y": (-5, 5),
    }
    for sid, (lo, hi) in range_checks.items():
        row = spark.sql(f"""
            SELECT MIN(value) AS mn, MAX(value) AS mx
            FROM precursor.bronze.fred_raw
            WHERE series_id = '{sid}' AND value IS NOT NULL
        """).collect()[0]
        mn, mx = row["mn"], row["mx"]
        in_range = (mn is not None) and (lo <= mn) and (mx <= hi)
        checks.append(_check(
            f"{sid}: values in [{lo}, {hi}]",
            in_range,
            f"min={mn:.4f}  max={mx:.4f}" if mn is not None else "no data",
        ))

    return checks

## Cell 8 — Main execution

In [0]:
_bootstrap_start = datetime.now()
logger.info("=== precursor.00_bootstrap_fred START at %s ===", _bootstrap_start.isoformat())

_failed_series: list[str] = []

try:
    _client   = get_fred_client()
    _combined = fetch_all_series(_client)
    write_fred_to_delta(_combined)

    _checks   = run_validation()

    _elapsed  = (datetime.now() - _bootstrap_start).total_seconds() / 60
    logger.info("=== precursor.00_bootstrap_fred END — %.2f min total ===", _elapsed)

    _passed_n = sum(1 for _, p, _ in _checks if p)
    _failed_n = len(_checks) - _passed_n

    _series_summary = (
        spark.sql("""
            SELECT series_id, COUNT(*) AS rows, MIN(date) AS min_date, MAX(date) AS max_date
            FROM precursor.bronze.fred_raw
            GROUP BY series_id
            ORDER BY series_id
        """).toPandas()
    )
    _total_rows = int(_series_summary["rows"].sum())

    print("=" * 60)
    print("  PRECURSOR — FRED BOOTSTRAP SUMMARY")
    print("=" * 60)
    for _, row in _series_summary.iterrows():
        print(f"  {row['series_id']:<10} {row['rows']:>7,} rows   {row['min_date']} → {row['max_date']}")
    print("-" * 60)
    print(f"  Total rows written : {_total_rows:,}")
    print(f"  Elapsed time       : {_elapsed:.2f} min")
    print("=" * 60)
    print("  VALIDATION RESULTS")
    print("=" * 60)
    for name, passed, detail in _checks:
        status = "PASS" if passed else "FAIL"
        line   = f"  [{status}] {name}"
        if detail:
            line += f"\n         {detail}"
        print(line)
    print("-" * 60)
    print(f"  {_passed_n} passed  /  {_failed_n} failed")
    if _failed_n > 0:
        print("  WARNING: validation failures detected — review logs above.")
    print("=" * 60)

except Exception as exc:
    logger.error("Bootstrap failed: %s", exc, exc_info=True)

INFO:precursor.fred_bootstrap:=== precursor.00_bootstrap_fred START at 2026-05-01T14:05:28.385467 ===
INFO:precursor.fred_bootstrap:FRED client initialised successfully.
INFO:precursor.fred_bootstrap:Fetching DFF — Fed Funds Rate (daily)
INFO:precursor.fred_bootstrap:DFF: 2311 observations fetched from 2020-01-01 to 2026-04-29
INFO:precursor.fred_bootstrap:Fetching UNRATE — Unemployment Rate (monthly)
INFO:precursor.fred_bootstrap:UNRATE: 75 observations fetched from 2020-01-01 to 2026-03-01
INFO:precursor.fred_bootstrap:UNRATE: forward-filled from 75 monthly rows to 2312 daily rows
INFO:precursor.fred_bootstrap:Fetching CPIAUCSL — CPI Inflation (monthly)
INFO:precursor.fred_bootstrap:CPIAUCSL: 75 observations fetched from 2020-01-01 to 2026-03-01
INFO:precursor.fred_bootstrap:CPIAUCSL: forward-filled from 75 monthly rows to 2312 daily rows
INFO:precursor.fred_bootstrap:Fetching T10Y2Y — 10Y-2Y Yield Spread (daily)
INFO:precursor.fred_bootstrap:T10Y2Y: 1652 observations fetched from 20

  PRECURSOR — FRED BOOTSTRAP SUMMARY
  CPIAUCSL     2,312 rows   2020-01-01 → 2026-04-30
  DFF          2,311 rows   2020-01-01 → 2026-04-29
  T10Y2Y       1,652 rows   2020-01-01 → 2026-04-30
  UNRATE       2,312 rows   2020-01-01 → 2026-04-30
  VIXCLS       1,652 rows   2020-01-01 → 2026-04-30
------------------------------------------------------------
  Total rows written : 10,239
  Elapsed time       : 0.43 min
  VALIDATION RESULTS
  [FAIL] Exactly 6 distinct series_ids
         5 found
  [PASS] All 6 series have data
         VIXCLS=1652, T10Y2Y=1652, CPIAUCSL=2312, UNRATE=2312, DFF=2311
  [PASS] T10Y2Y: latest date within 5 days
         latest = 2026-04-30
  [PASS] VIXCLS: latest date within 5 days
         latest = 2026-04-30
  [PASS] DFF: latest date within 5 days
         latest = 2026-04-29
  [PASS] T10Y2Y: no gaps > 5 days
         max gap = 3 days
  [PASS] VIXCLS: no gaps > 5 days
         max gap = 3 days
  [PASS] DFF: no gaps > 5 days
         max gap = 1 days
  [FAIL] 

In [0]:
from pyspark.sql.window import Window

# Detect which series need patching:
#   1. Any series in FRED_SERIES not present in the table at all
#   2. Any series that has null values in the table
_existing_series = set(
    spark.sql("SELECT DISTINCT series_id FROM precursor.bronze.fred_raw")
    .toPandas()["series_id"]
    .tolist()
)

_null_series = set(
    spark.sql("""
        SELECT DISTINCT series_id
        FROM precursor.bronze.fred_raw
        WHERE value IS NULL
    """)
    .toPandas()["series_id"]
    .tolist()
)

_missing_series = set(FRED_SERIES.keys()) - _existing_series
_patch_targets  = _missing_series | _null_series

if not _patch_targets:
    print("No series need patching — all series present and null-free.")
else:
    logger.info("Patch targets: %s", sorted(_patch_targets))

    _patch_client = get_fred_client()
    _patch_frames = []

    for _sid in sorted(_patch_targets):
        _desc = FRED_SERIES[_sid]
        logger.info("Re-fetching %s — %s", _sid, _desc)
        _df = fetch_fred_series(_patch_client, _sid, _desc, START_DATE, END_DATE)
        if _df is None:
            logger.error("%s: still failed after retry — skipping.", _sid)
            continue
        if _sid in _MONTHLY_SERIES:
            _df = forward_fill_to_daily(_df)
        _patch_frames.append(_df)

    if _patch_frames:
        _patch_combined = pd.concat(_patch_frames, ignore_index=True)
        _patch_combined["date"]        = pd.to_datetime(_patch_combined["date"])
        _patch_combined["ingested_at"] = pd.to_datetime(_patch_combined["ingested_at"])
        _patch_combined["value"]       = _patch_combined["value"].astype(float)

        _patch_spark    = spark.createDataFrame(
            _patch_combined[["series_id","description","date","value","frequency","is_forward_filled","ingested_at"]],
            schema=_FRED_RAW_SCHEMA,
        )
        _patch_ids_sql  = ",".join(f"'{s}'" for s in _patch_targets)

        # Union with existing rows for these series, deduplicate on (series_id, date)
        try:
            _patch_existing = spark.sql(f"""
                SELECT * FROM precursor.bronze.fred_raw
                WHERE series_id IN ({_patch_ids_sql})
            """)
            _patch_spark = _patch_existing.union(_patch_spark)
        except Exception:
            pass

        _patch_deduped = (
            _patch_spark
            .withColumn(
                "rn",
                F.row_number().over(
                    Window.partitionBy("series_id", "date").orderBy(F.desc("ingested_at"))
                )
            )
            .filter(F.col("rn") == 1)
            .drop("rn")
        )

        (
            _patch_deduped.write
            .format("delta")
            .mode("overwrite")
            .option("replaceWhere", f"series_id IN ({_patch_ids_sql})")
            .option("mergeSchema", "true")
            .saveAsTable("precursor.bronze.fred_raw")
        )

        logger.info("Patch complete for: %s", sorted(_patch_targets))

        for _sid in sorted(_patch_targets):
            _r = spark.sql(f"""
                SELECT COUNT(*) AS n,
                       SUM(CASE WHEN value IS NULL THEN 1 ELSE 0 END) AS nulls,
                       MIN(date) AS mn, MAX(date) AS mx
                FROM precursor.bronze.fred_raw
                WHERE series_id = '{_sid}'
            """).collect()[0]
            print(f"  {_sid}: {_r['n']} rows  {_r['mn']} → {_r['mx']}  nulls={_r['nulls']}")
    else:
        logger.error("Patch attempted but no data returned for any target series.")

INFO:precursor.fred_bootstrap:Patch targets: ['CPIAUCSL', 'M2SL', 'T10Y2Y', 'UNRATE', 'VIXCLS']
INFO:precursor.fred_bootstrap:FRED client initialised successfully.
INFO:precursor.fred_bootstrap:Re-fetching CPIAUCSL — CPI Inflation (monthly)
INFO:precursor.fred_bootstrap:CPIAUCSL: 75 observations fetched from 2020-01-01 to 2026-03-01
INFO:precursor.fred_bootstrap:CPIAUCSL: forward-filled from 75 monthly rows to 2312 daily rows
INFO:precursor.fred_bootstrap:Re-fetching M2SL — M2 Money Supply (monthly)
INFO:precursor.fred_bootstrap:M2SL: 75 observations fetched from 2020-01-01 to 2026-03-01
INFO:precursor.fred_bootstrap:M2SL: forward-filled from 75 monthly rows to 2312 daily rows
INFO:precursor.fred_bootstrap:Re-fetching T10Y2Y — 10Y-2Y Yield Spread (daily)
INFO:precursor.fred_bootstrap:T10Y2Y: 1652 observations fetched from 2020-01-01 to 2026-04-30
INFO:precursor.fred_bootstrap:Re-fetching UNRATE — Unemployment Rate (monthly)
INFO:precursor.fred_bootstrap:UNRATE: 75 observations fetched 

  CPIAUCSL: 2312 rows  2020-01-01 → 2026-04-30  nulls=31
  M2SL: 2312 rows  2020-01-01 → 2026-04-30  nulls=0
  T10Y2Y: 1652 rows  2020-01-01 → 2026-04-30  nulls=69
  UNRATE: 2312 rows  2020-01-01 → 2026-04-30  nulls=31
  VIXCLS: 1652 rows  2020-01-01 → 2026-04-30  nulls=33


In [0]:
_before = spark.table("precursor.bronze.fred_raw").count()

(
    spark.table("precursor.bronze.fred_raw")
    .filter(F.col("value").isNotNull())
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("precursor.bronze.fred_raw")
)

_after = spark.table("precursor.bronze.fred_raw").count()
logger.info("Dropped %d null-value rows. Rows before: %d  after: %d", _before - _after, _before, _after)
print(f"  Rows before : {_before:,}")
print(f"  Rows after  : {_after:,}")
print(f"  Rows dropped: {_before - _after:,}")

INFO:precursor.fred_bootstrap:Dropped 164 null-value rows. Rows before: 12551  after: 12387


  Rows before : 12,551
  Rows after  : 12,387
  Rows dropped: 164


In [0]:
_final_checks = run_validation()
_passed_n = sum(1 for _, p, _ in _final_checks if p)
_failed_n  = len(_final_checks) - _passed_n

_series_summary = spark.sql("""
    SELECT series_id, COUNT(*) AS rows, MIN(date) AS min_date, MAX(date) AS max_date
    FROM precursor.bronze.fred_raw
    GROUP BY series_id
    ORDER BY series_id
""").toPandas()

print("=" * 60)
print("  PRECURSOR — FRED FINAL STATE")
print("=" * 60)
for _, row in _series_summary.iterrows():
    print(f"  {row['series_id']:<10} {row['rows']:>6,} rows   {row['min_date']} → {row['max_date']}")
print(f"  {'TOTAL':<10} {int(_series_summary['rows'].sum()):>6,} rows")
print("=" * 60)
print("  VALIDATION RESULTS")
print("=" * 60)
for name, passed, detail in _final_checks:
    status = "PASS" if passed else "FAIL"
    line   = f"  [{status}] {name}"
    if detail:
        line += f"\n         {detail}"
    print(line)
print("-" * 60)
print(f"  {_passed_n} passed  /  {_failed_n} failed")
if _failed_n > 0:
    print("  WARNING: validation failures detected — review logs above.")
print("=" * 60)

INFO:precursor.fred_bootstrap:[PASS] Exactly 6 distinct series_ids — 6 found
INFO:precursor.fred_bootstrap:[PASS] All 6 series have data — DFF=2311, CPIAUCSL=2281, M2SL=2312, T10Y2Y=1583, UNRATE=2281, VIXCLS=1619
INFO:precursor.fred_bootstrap:[PASS] T10Y2Y: latest date within 5 days — latest = 2026-04-30
INFO:precursor.fred_bootstrap:[PASS] VIXCLS: latest date within 5 days — latest = 2026-04-30
INFO:precursor.fred_bootstrap:[PASS] DFF: latest date within 5 days — latest = 2026-04-29
INFO:precursor.fred_bootstrap:[PASS] T10Y2Y: no gaps > 5 days — max gap = 4 days
INFO:precursor.fred_bootstrap:[PASS] VIXCLS: no gaps > 5 days — max gap = 4 days
INFO:precursor.fred_bootstrap:[PASS] DFF: no gaps > 5 days — max gap = 1 days
INFO:precursor.fred_bootstrap:[PASS] UNRATE: no nulls after forward fill — 0 nulls found
INFO:precursor.fred_bootstrap:[PASS] M2SL: no nulls after forward fill — 0 nulls found
INFO:precursor.fred_bootstrap:[PASS] CPIAUCSL: no nulls after forward fill — 0 nulls found
INFO

  PRECURSOR — FRED FINAL STATE
  CPIAUCSL    2,281 rows   2020-01-01 → 2026-04-30
  DFF         2,311 rows   2020-01-01 → 2026-04-29
  M2SL        2,312 rows   2020-01-01 → 2026-04-30
  T10Y2Y      1,583 rows   2020-01-02 → 2026-04-30
  UNRATE      2,281 rows   2020-01-01 → 2026-04-30
  VIXCLS      1,619 rows   2020-01-02 → 2026-04-30
  TOTAL      12,387 rows
  VALIDATION RESULTS
  [PASS] Exactly 6 distinct series_ids
         6 found
  [PASS] All 6 series have data
         DFF=2311, CPIAUCSL=2281, M2SL=2312, T10Y2Y=1583, UNRATE=2281, VIXCLS=1619
  [PASS] T10Y2Y: latest date within 5 days
         latest = 2026-04-30
  [PASS] VIXCLS: latest date within 5 days
         latest = 2026-04-30
  [PASS] DFF: latest date within 5 days
         latest = 2026-04-29
  [PASS] T10Y2Y: no gaps > 5 days
         max gap = 4 days
  [PASS] VIXCLS: no gaps > 5 days
         max gap = 4 days
  [PASS] DFF: no gaps > 5 days
         max gap = 1 days
  [PASS] UNRATE: no nulls after forward fill
         0 